# 🔬 AI 내부 들여다보기 — transformer_lens

**수술하려면 먼저 들여다봐야 한다.** AI의 '뇌 MRI' — 어떤 생각을 할 때 내부가 어떻게 켜지는지 본다.

> 이건 🧠 거부 방향(#5)·🎚️ 조종(#6) 수술의 **토대**예요. 내부를 봐야 어디를 만질지 안다.

**비유**: 모델이 답을 내기까지, 레이어를 거치며 **생각이 점점 또렷해진다.** 그 과정을 단계별로 펼쳐 본다.
**도구**: `transformer_lens` (해석가능성 표준 도구) · **모델**: gpt2-small(구조가 깨끗해 학습용 최적) · **실행**: 무료 Colab(T4, GPU 없어도 됨)

## 1단계 — 설치

In [ ]:
!pip -q install transformer_lens matplotlib

## 2단계 — 모델 로드 (HookedTransformer = 내부가 다 열린 모델)

In [ ]:
import torch
from transformer_lens import HookedTransformer
model = HookedTransformer.from_pretrained("gpt2")
print("레이어:", model.cfg.n_layers, "· 헤드:", model.cfg.n_heads, "· 차원:", model.cfg.d_model)

## 3단계 — 활성화 통째로 잡기 (run_with_cache)

한 문장을 넣고, **모든 레이어의 내부 활성화**를 캐시에 담는다.

In [ ]:
prompt = "The Eiffel Tower is located in the city of"
tokens = model.to_tokens(prompt)
logits, cache = model.run_with_cache(tokens)
pred = model.to_string(logits[0, -1].argmax())
print("최종 예측 다음 단어:", repr(pred))
print("캐시에 담긴 활성화 종류 예:", [k for k in list(cache.keys())[:6]])

## 4단계 — 🔭 Logit Lens: 레이어마다 '지금 떠오른 답'

각 레이어의 residual stream을 **마지막 출력층으로 곧장 디코딩**해, 모델이 층을 지나며 답을 어떻게 좁혀가는지 본다. (초반엔 엉뚱 → 후반에 'Paris'로 수렴하면 성공)

In [ ]:
import matplotlib.pyplot as plt
preds = []
for l in range(model.cfg.n_layers):
    resid = cache["resid_post", l][0, -1]          # 마지막 토큰의 그 레이어 출력
    resid = model.ln_final(resid)                    # 최종 정규화
    layer_logits = resid @ model.W_U                 # 출력층으로 디코딩
    preds.append(model.to_string(layer_logits.argmax()))
for l, p in enumerate(preds):
    print(f"레이어 {l:2d} → {repr(p)}")

## 5단계 — 👁️ Attention 패턴: 어느 단어를 보고 있나

한 레이어·한 헤드가 **어떤 단어에 주목**하는지 행렬로 본다. (대각선·특정 토큰에 쏠리면 그 헤드의 '역할'이 보임)

In [ ]:
layer, head = 0, 0
pattern = cache["pattern", layer][0, head]           # [토큰, 토큰]
labels = model.to_str_tokens(prompt)
plt.figure(figsize=(6,5))
plt.imshow(pattern.cpu().numpy(), cmap="viridis")
plt.xticks(range(len(labels)), labels, rotation=90); plt.yticks(range(len(labels)), labels)
plt.title(f"Attention — 레이어 {layer}, 헤드 {head}"); plt.colorbar(); plt.show()

## 6단계 — 보너스: 레이어별 '활동량'

각 레이어 출력의 크기(norm)를 재 — 어느 층에서 생각이 많이 바뀌는지.

In [ ]:
norms = [cache["resid_post", l][0, -1].norm().item() for l in range(model.cfg.n_layers)]
plt.figure(figsize=(7,3)); plt.plot(range(model.cfg.n_layers), norms, marker="o")
plt.title("레이어별 residual stream 크기"); plt.xlabel("레이어"); plt.ylabel("norm"); plt.show()

---
## 🎓 무슨 일이 일어난 건가

- **Logit Lens**: 모델은 한 번에 답을 내지 않는다 — 레이어를 지나며 **생각이 점점 정답으로 수렴**한다.
- **Attention**: 각 헤드는 특정 단어 관계에 **주목하는 역할**을 나눠 맡는다.
- 이렇게 내부를 '읽을' 수 있으면, **어느 방향·어느 레이어를 만질지** 알게 된다 → 거부 방향(#5)·조종(#6)의 토대.

## 📚 레퍼런스
| 주제 | 논문/자료 | 링크 |
| --- | --- | --- |
| 표현 공학(해석) | Representation Engineering (Zou) | 2310.01405 |
| 단의미성·SAE | Anthropic Monosemanticity | transformer-circuits.pub |
| 도구 | transformer_lens (Nanda) | github.com/TransformerLensOrg |